In [ ]:
%pip install "librosa==1.0.0" --quiet

# Exercise — Module M4-CNN: spectrogram CNN บน ESC-50

ทำบน Kaggle (GPU T4, Internet On) — เติมโค้ดตรง `# TODO` แล้วตอบคำถามใน
markdown ที่กำหนด รันเซลล์ตามลำดับจากบนลงล่าง

**ที่มา:** ESC-50 ([github.com/karoldvl/ESC-50](https://github.com/karoldvl/ESC-50))
— license **CC BY-NC 3.0 (non-commercial)** ห้ามแจกต่อเชิงพาณิชย์

## 0. เตรียมข้อมูล + baseline + เทรน CNN (ให้มาแล้ว — รันผ่าน)

เตรียมข้อมูลให้จบก่อน (รวมเวลา ~15–20 นาที: ดาวน์โหลด ESC-50 ~600 MB →
แปลง spectrogram → RF baseline → เทรน CNN 30 epochs) แล้วตัวเลขจากช่วงนี้
เป็นฐานของทุกข้อ

In [ ]:
import os  # noqa: E402

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import librosa  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
from joblib import Parallel, delayed  # noqa: E402
from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import accuracy_score  # noqa: E402
from torch.utils.data import DataLoader, TensorDataset  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
SR = 22050

In [ ]:
from pathlib import Path  # noqa: E402

import requests  # noqa: E402
import zipfile  # noqa: E402

ESC50_URL = "https://github.com/karoldvl/ESC-50/archive/master.zip"


def ensure_esc50() -> Path:
    """ดาวน์โหลด/แตกไฟล์ ESC-50 ครั้งเดียว แล้วคืน path ของโฟลเดอร์."""
    on_kaggle = Path("/kaggle").exists()
    base = (
        Path("/kaggle/tmp") if on_kaggle
        else Path.home() / ".cache" / "esc50"
    )
    found = base / "ESC-50-master"
    if (found / "meta" / "esc50.csv").exists():
        print("ใช้ ESC-50 ที่มีอยู่แล้ว:", found)
        return found
    base.mkdir(parents=True, exist_ok=True)
    zpath = base / "esc50-master.zip"
    if not zpath.exists():
        print("ดาวน์โหลด ESC-50 (~600 MB) — ใช้เวลา 2–5 นาที")
        with requests.get(ESC50_URL, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(zpath, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(base)
    print("แตกไฟล์แล้วที่:", found)
    return found


ESC50 = ensure_esc50()


In [ ]:
meta = pd.read_csv(ESC50 / "meta" / "esc50.csv")
id2cat = (
    meta.drop_duplicates("target")
    .sort_values("target")["category"]
    .tolist()
)
paths = [ESC50 / "audio" / f for f in meta["filename"]]
y_all = meta["target"].to_numpy()
folds = meta["fold"].to_numpy()
print("จำนวนคลิป:", len(meta))

In [ ]:
def mel_row(path) -> np.ndarray:
    """คลิปเดียว → log-mel spectrogram (128, 216) float32."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128)
    return librosa.power_to_db(S).astype(np.float32)


mels = np.asarray(Parallel(n_jobs=-1)(
    delayed(mel_row)(p) for p in paths
))
tr, te = folds != 5, folds == 5
Xtr = torch.from_numpy(mels[tr]).unsqueeze(1)
Xte = torch.from_numpy(mels[te]).unsqueeze(1)
ytr = torch.from_numpy(y_all[tr])
yte = torch.from_numpy(y_all[te])
mean, std = Xtr.mean(), Xtr.std()
Xtr_n = (Xtr - mean) / std
Xte_n = (Xte - mean) / std
print("train:", tuple(Xtr_n.shape), "| test (fold 5):",
      tuple(Xte_n.shape))

In [ ]:
def mfcc_row(path) -> np.ndarray:
    """คลิปเดียว → เวกเตอร์ 40 ตัวเลข (mean+std ของ MFCC 20 ตัว)."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20)
    return np.concatenate([m.mean(axis=1), m.std(axis=1)])


X_mfcc = np.asarray(Parallel(n_jobs=-1)(
    delayed(mfcc_row)(p) for p in paths
))
rf = RandomForestClassifier(
    n_estimators=200, n_jobs=-1, random_state=SEED
).fit(X_mfcc[tr], y_all[tr])
acc_rf5 = accuracy_score(y_all[te], rf.predict(X_mfcc[te]))
print(f"RF บน MFCC — accuracy (fold 5) = {acc_rf5:.4f}")

In [ ]:
class SmallCNN(nn.Module):
    """CNN จิ๋ว 3 ชั้น conv — อ่าน spectrogram (1, in_mels, in_frames)."""

    def __init__(self, in_mels=128, in_frames=216, n_classes=50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(
                64 * (in_mels // 8) * (in_frames // 8), n_classes
            ),
        )

    def forward(self, x):
        return self.net(x)


def train_cnn(Xa, ya, epochs=30, bs=32, seed=0, log_every=5):
    """เทรน CNN ด้วย Adam บน GPU — คืนโมเดลที่เทรนแล้ว."""
    torch.manual_seed(seed)
    model = SmallCNN(
        in_mels=Xa.shape[2], in_frames=Xa.shape[3]
    ).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(
        TensorDataset(Xa, ya), batch_size=bs, shuffle=True,
        generator=g,
    )
    for ep in range(epochs):
        model.train()
        run = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            run += loss.item()
        if log_every and (ep + 1) % log_every == 0:
            print(f"  epoch {ep + 1}/{epochs}: "
                  f"loss={run / len(dl):.4f}")
    return model


def cnn_accuracy(model, Xa, ya):
    model.eval()
    with torch.no_grad():
        pred = model(Xa.to(DEVICE)).argmax(1).cpu().numpy()
    return accuracy_score(ya.numpy(), pred)

เทรน CNN 30 epochs (ไว้อ้างอิงข้อ 1 — จด accuracy ที่ได้ด้วย):

In [ ]:
cnn = train_cnn(Xtr_n, ytr, epochs=30)
acc_cnn = cnn_accuracy(cnn, Xte_n, yte)
print(f"CNN 30 epochs — accuracy (fold 5) = {acc_cnn:.4f}")

## ข้อ 1 — เทรนนานขึ้นช่วยไหม? (60 epochs)

เทรนอีกโมเดลด้วย `epochs=60` (~5–10 นาที) — ดูทั้ง loss ระหว่างเทรนและ
accuracy ปลายทาง แล้วตอบว่า 30 เพิ่มเป็น 60 ได้อะไร

In [ ]:
# TODO 1: เทรน 60 epochs แล้ววัด accuracy บน fold 5
cnn60 = ...  # noqa: F841
acc60 = ...  # noqa: F841

คำตอบข้อ 1: (60 epochs ได้ accuracy ต่างจาก 30 มากไหม — loss หลัง
epoch 30 เป็นอย่างไร นั่นบอกอะไรเกี่ยวกับ overfit)

## ข้อ 2 — ลดความละเอียด spectrogram: n_mels=64

spectrogram 128 แถบ mel อาจละเอียดกว่าที่จำเป็น — แปลงใหม่ด้วย `n_mels=64`
(ภาพเล็กลงครึ่งหนึ่ง) แล้วเทรน CNN ใหม่ 30 epochs — `SmallCNN` รับ
`in_mels` เป็น argument อยู่แล้ว

In [ ]:
# TODO 2: แปลง spectrogram ใหม่ด้วย n_mels=64 → normalize → เทรนใหม่
# ใบ้: ทำซ้ำขั้นตอนเดียวกันกับ MEL_PREP แต่เปลี่ยน n_mels ใน mel_row
# (และห้ามลืม: tensor ใหม่ = (64, 216) — SmallCNN รับ in_mels=64 เอง
# ผ่าน Xa.shape)

คำตอบข้อ 2: (accuracy กับ n_mels=64 ต่างจาก 128 แค่ไหน — แลกกับ
หน่วยความจำ/เวลาที่ประหยัดได้ คุ้มไหม)

## ข้อ 3 — CNN สับสนคู่ไหน

ใช้โมเดล 30 epochs ที่ให้มา: ทำนาย fold 5 แล้วนับคู่สับสนบ่อยสุดด้วย
`Counter` — เทียบกับที่ RF สับสน (จาก book ของครึ่งแรก) ว่าคู่เดิมไหม

In [ ]:
from collections import Counter  # noqa: E402

# TODO 3: ทำนาย fold 5 ด้วย cnn แล้วนับคู่ผิด
# ใบ้: pred = cnn(Xte_n.to(DEVICE)).argmax(1).cpu().numpy()
pred = ...  # noqa: F841

wrong = Counter(
    (id2cat[t], id2cat[p])
    for t, p in zip(y_all[te], pred) if t != p
)
print(wrong.most_common(8))

คำตอบข้อ 3: (คู่สับสนบ่อยสุดของ CNN — เหมือน/ต่างจาก RF อย่างไร
และสมเหตุสมผลทางเสียงไหม)

## ข้อ 4 — กรอกตาราง bake-off ของคุณเอง

จากตัวเลขที่คุณรันได้จริงทั้งหมดใน notebook นี้ กรอกตารางด้านล่างแล้วตอบ
คำถามท้าย

| วิธี | input | accuracy (fold 5 — คุณรันได้) |
|---|---|---|
| RF บน MFCC | ตาราง 40 ตัวเลข | |
| CNN 30 epochs (n_mels=128) | ภาพ 128×216 | |
| CNN 60 epochs (ข้อ 1) | ภาพ 128×216 | |
| CNN 30 epochs (n_mels=64, ข้อ 2) | ภาพ 64×216 | |

**ตอบ:** ตัวเลขที่เพิ่มขึ้นจาก RF → CNN มาจากอะไร (ข้อจำกัดของ MFCC
40 ตัวเลข vs ข้อมูลเต็มใน spectrogram)

## ข้อ 5 — ตัดสินใจแบบนักเมโทรโลยี (ตอบใน markdown)

โจทย์จริง: ทีมคุณอัดเสียงเครื่องจักรมา 200 คลิป (5 วินาที) แบ่ง "ปกติ /
ผิดปกติ" — จะ deploy บนเครื่อง monitor ข้างเครื่องจักรที่ไม่มี GPU
**เลือกวิธีและอธิบาย 2–3 เหตุผลที่ชี้เฉพาะโจทย์นี้** (ใบ้: ปริมาณข้อมูล,
เครื่องปลายทาง, ความสามารถในการตรวจสอบ)

คำตอบข้อ 5:



## สรุป (เติมหลังทำครบ)

- เทรนนานขึ้น (30 → 60 epochs) ให้ผล: ...
- spectrogram หยาบลง (128 → 64 mels) ให้ผล: ...
- CNN แพ้/ชนะ RF ตรง: ...
- สิ่งที่จะไปดูต่อ (M5) คือ: ...